In [21]:
api_key="pcsk_UTiNZ_629xu2LJm6rLzcQZwrCkAHzNiwrLHEr31d1Z2Zq6aNsg6LbzaxxmcpYYaXqUzUT"

In [22]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [23]:
import os
from pinecone import Pinecone,ServerlessSpec
index_name="hybrid-search-langchain-pinecone"

## Initialize the Pinecone client:
pc=Pinecone(api_key=api_key)

## Create an Index:
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric='dotproduct',
        spec=ServerlessSpec(cloud='aws',region="us-east-1")
    )

In [24]:
index=pc.Index(index_name)
index

Index(host='https://hybrid-search-langchain-pinecone-nyubid2.svc.aped-4627-b74a.pinecone.io')

In [25]:
## Vector Embedding and sparse matrix:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3316.39it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [26]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder=BM25Encoder().default()
bm25_encoder

In [27]:
sentences=["In 2023, I visited Paris",
            "In 2022,I visted New York",
            "In 2021, I visited New Orleans"]

bm25_encoder.fit(sentences)

bm25_encoder.dump("bm25_values.json")

bm25_encoder=BM25Encoder().load("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00, 3018.93it/s]


In [28]:
## Creating Retriever:
retriever=PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index)
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000001BB005DDFF0>, index=Index(host='https://hybrid-search-langchain-pinecone-nyubid2.svc.aped-4627-b74a.pinecone.io'))

In [29]:
retriever.add_texts(
    ["In 2023, I visited Paris",
            "In 2022,I visted New York",
            "In 2021, I visited New Orleans"]
)

  0%|          | 0/1 [00:00<?, ?it/s]


TypeError: Index.upsert() takes 1 positional argument but 2 positional arguments (and 1 keyword-only argument) were given